# GateMind AI — Week 3 V1 Data Pipeline
**UTM AI Engineering | Topic 3 | Selected Pathway F (Time-Series)**  
This notebook demonstrates the required shared stages with the lecturer's **synthetic energy CSV as a structural practice proxy**. It is **not smart-lock training data**, and no security classification results are claimed. The eventual data source is consented smart-lock event logs and/or permitted ambient-sensor public data.

**Run order:** Workspace → upload course ZIP → verify source → inspect → prepare → validate → lineage → reusable V1 pipeline → repeatability check. Use *Runtime → Run all* after upload. No model API key is required.

## Cell 1 — workspace and canonical paths

In [1]:
from pathlib import Path
import json, hashlib, zipfile, shutil
from datetime import datetime, timezone
import pandas as pd
BASE_DIR = Path('/content/week3_project') if Path('/content').exists() else Path('/mnt/data/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW=RAW_DIR; BASE=BASE_DIR; OUT=PROCESSED_DIR
print('Raw:',RAW_DIR,'| Prepared:',PROCESSED_DIR)

Raw: /content/week3_project/data/raw | Prepared: /content/week3_project/data/processed


## Cell 2 — upload and extract the optional course ZIP
Upload `Topic_3_Dummy_Datasets.zip` in Colab using the Files sidebar or the prompted uploader. You can rerun safely after a runtime reset.

In [2]:
ZIP = RAW_DIR / 'Topic_3_Dummy_Datasets.zip'
if not ZIP.exists():
    try:
        from google.colab import files
        print('Please select Topic_3_Dummy_Datasets.zip')
        uploaded = files.upload()
        assert 'Topic_3_Dummy_Datasets.zip' in uploaded, 'Select the specified course ZIP.'
        ZIP.write_bytes(uploaded['Topic_3_Dummy_Datasets.zip'])
    except ImportError:
        # For local testing only: user-provided archive path
        local = Path('/mnt/data/Topic_3_Dummy_Datasets.zip')
        assert local.exists(), 'Place the course ZIP in the raw folder.'
        ZIP.write_bytes(local.read_bytes())
with zipfile.ZipFile(ZIP) as z:
    for entry in z.infolist():
        target=(RAW_DIR / entry.filename).resolve()
        assert target.is_relative_to(RAW_DIR.resolve()), 'Unsafe archive path'
        z.extract(entry, RAW_DIR)
print('Extracted course pack; total raw files:',sum(p.is_file() for p in RAW_DIR.rglob('*')))

Please select Topic_3_Dummy_Datasets.zip


Saving Topic_3_Dummy_Datasets.zip to Topic_3_Dummy_Datasets.zip
Extracted course pack; total raw files: 17


## Cell 3 — verify explicit input and inspect two representative subsets

In [3]:
RELATIVE_PATH='topic3_dummy_datasets/06_timeseries/energy_readings_demo.csv'
SOURCE=RAW_DIR / RELATIVE_PATH
assert SOURCE.is_file(), f'Missing: {SOURCE}'
all_files=sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Verified raw file count:', len(all_files))
print('Selected:', SOURCE.relative_to(RAW_DIR), 'bytes:',SOURCE.stat().st_size)
original=pd.read_csv(SOURCE)
assert {'timestamp','sensor_id','energy_kwh'}.issubset(original.columns)
assert not original.empty
print('Schema:',original.dtypes.to_dict(), '| Shape:',original.shape)
print('Subset 1:');display(original.head(3))
print('Subset 2:');display(original.tail(3))
print('Missing:',original.isna().sum().to_dict(),'| Duplicate rows:',int(original.duplicated().sum()))

Verified raw file count: 17
Selected: topic3_dummy_datasets/06_timeseries/energy_readings_demo.csv bytes: 185
Schema: {'timestamp': dtype('O'), 'sensor_id': dtype('O'), 'energy_kwh': dtype('float64')} | Shape: (6, 3)
Subset 1:


,timestamp,sensor_id,energy_kwh
0,2026-10-01 00:00,M01,1.2
1,2026-10-01 01:00,M01,1.4
2,2026-10-01 03:00,M01,1.8


Subset 2:


,timestamp,sensor_id,energy_kwh
3,2026-10-01 02:00,M01,1.6
4,2026-10-01 04:00,M01,NaN
5,2026-10-01 05:00,M01,9.8


Missing: {'timestamp': 0, 'sensor_id': 0, 'energy_kwh': 1} | Duplicate rows: 0


## Cell 4 / 4A — source checksums and lineage registry

In [4]:
def source_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(65536),b''):h.update(chunk)
    return h.hexdigest()
def select_source(relative_path):
    p=RAW_DIR/relative_path
    if not p.is_file():raise FileNotFoundError(p)
    return p
lineage_entries={}
def register_lineage(source,output,processing):
    source=Path(source);output=Path(output)
    assert source.is_file() and output.is_file() and output.stat().st_size>0
    assert processing.strip()
    lineage_entries[(str(source),str(output))]={'source':source,'output':output,'processing':processing}
print('Helpers ready; source fingerprint:',source_hash(SOURCE)[:20])

Helpers ready; source fingerprint: 6394aa3d01d35f4df554


## Cell 5F — selected time-series preparation
Do not run unrelated Pathways A–E. Preserve the missing reading; it must not be silently imputed. Change path/schema for future genuine GateMind logs.

In [5]:
SOURCE=select_source(RELATIVE_PATH)
raw=pd.read_csv(SOURCE)
print('Original rows:',len(raw),'| Out-of-order example:');display(raw[['timestamp','energy_kwh']].head())
parsed=pd.to_datetime(raw['timestamp'],errors='coerce')
invalid_count=int(parsed.isna().sum())
prepared=raw.copy();prepared['timestamp']=parsed
prepared=prepared.dropna(subset=['timestamp']).sort_values(['timestamp','sensor_id'],kind='stable').reset_index(drop=True)
prepared['timestamp']=prepared['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
PREPARED_PATH=PROCESSED_DIR/'prepared_timeseries.csv'
prepared.to_csv(PREPARED_PATH,index=False)
print('Before:',len(raw),'After:',len(prepared),'| Invalid timestamps:',invalid_count)
print('Preserved missing energy values:',int(prepared['energy_kwh'].isna().sum()))
display(prepared)

Original rows: 6 | Out-of-order example:


,timestamp,energy_kwh
0,2026-10-01 00:00,1.2
1,2026-10-01 01:00,1.4
2,2026-10-01 03:00,1.8
3,2026-10-01 02:00,1.6
4,2026-10-01 04:00,NaN


Before: 6 After: 6 | Invalid timestamps: 0
Preserved missing energy values: 1


,timestamp,sensor_id,energy_kwh
0,2026-10-01T00:00:00,M01,1.2
1,2026-10-01T01:00:00,M01,1.4
2,2026-10-01T02:00:00,M01,1.6
3,2026-10-01T03:00:00,M01,1.8
4,2026-10-01T04:00:00,M01,NaN
5,2026-10-01T05:00:00,M01,9.8


## Step B3 — validation, source provenance and Part B manifest

In [6]:
saved=pd.read_csv(PREPARED_PATH)
assert len(saved)==len(prepared) and saved.columns.tolist()==prepared.columns.tolist()
assert pd.to_datetime(saved['timestamp']).is_monotonic_increasing
assert saved['energy_kwh'].isna().sum()==raw['energy_kwh'].isna().sum()
assert source_hash(PREPARED_PATH)
PROCESSING='Parse timestamps; omit invalid times; stable chronological sort; preserve missing sensor reading; no resampling or imputation'
register_lineage(SOURCE,PREPARED_PATH,PROCESSING)
records=[{'source':str(e['source'].relative_to(RAW_DIR)),'source_sha256':source_hash(e['source']),
          'output':str(e['output'].relative_to(BASE_DIR)),'output_sha256':source_hash(e['output']),
          'processing':e['processing'],'input_rows':len(raw),'output_rows':len(prepared),'invalid_timestamps':invalid_count}
         for e in lineage_entries.values()]
MANIFEST_PATH=BASE_DIR/'data/week3_manifest.json'
MANIFEST_PATH.write_text(json.dumps({'pipeline_version':'week3-v1','generated_at_utc':datetime.now(timezone.utc).isoformat(),'records':records},indent=2))
check=json.loads(MANIFEST_PATH.read_text())
assert check['records'][0]['source_sha256']==source_hash(SOURCE)
assert check['records'][0]['output_sha256']==source_hash(PREPARED_PATH)
print('COMMON LINEAGE VALIDATION: PASS; records:',len(records),'| file:',MANIFEST_PATH)

COMMON LINEAGE VALIDATION: PASS; records: 1 | file: /content/week3_project/data/week3_manifest.json


## Cell 7 / 8F — reusable executable V1 pipeline and run manifest

In [7]:
def run_pipeline_timeseries(relative_path=RELATIVE_PATH):
    source=select_source(relative_path)
    incoming=pd.read_csv(source)
    required={'timestamp','sensor_id','energy_kwh'}
    if not required.issubset(incoming.columns):raise ValueError(f'Missing columns: {required-set(incoming.columns)}')
    times=pd.to_datetime(incoming['timestamp'],errors='coerce')
    data=incoming.copy();data['timestamp']=times
    invalid=int(data['timestamp'].isna().sum())
    data=data.dropna(subset=['timestamp']).sort_values(['timestamp','sensor_id'],kind='stable').reset_index(drop=True)
    if data.empty:raise ValueError('No valid records')
    data['timestamp']=data['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
    target_dir=BASE_DIR/'data/pipeline_v1/timeseries';target_dir.mkdir(parents=True,exist_ok=True)
    target=target_dir/'prepared_timeseries.csv';data.to_csv(target,index=False)
    run={'pipeline_version':'week3-v1-timeseries','source_path':relative_path,'source_sha256':source_hash(source),
         'output_path':str(target.relative_to(BASE_DIR)),'output_sha256':source_hash(target),
         'input_rows':len(incoming),'output_rows':len(data),'invalid_timestamps':invalid,
         'missing_readings':int(data['energy_kwh'].isna().sum()),'processing':PROCESSING,
         'generated_at_utc':datetime.now(timezone.utc).isoformat(),'status':'SUCCESS'}
    manifest_path=target_dir/'pipeline_manifest.json';manifest_path.write_text(json.dumps(run,indent=2))
    return target,manifest_path,run
PIPELINE_OUT,PIPELINE_MANIFEST,RUN1=run_pipeline_timeseries()
print('V1 pipeline SUCCESS:',RUN1['input_rows'],'→',RUN1['output_rows'])
print('Output:',PIPELINE_OUT,'| Manifest:',PIPELINE_MANIFEST)
display(pd.read_csv(PIPELINE_OUT).head(3))

V1 pipeline SUCCESS: 6 → 6
Output: /content/week3_project/data/pipeline_v1/timeseries/prepared_timeseries.csv | Manifest: /content/week3_project/data/pipeline_v1/timeseries/pipeline_manifest.json


,timestamp,sensor_id,energy_kwh
0,2026-10-01T00:00:00,M01,1.2
1,2026-10-01T01:00:00,M01,1.4
2,2026-10-01T02:00:00,M01,1.6


## Cell 9 — repeatability & provenance smoke test
Run the same pipeline twice; generated timestamps can differ. This tests source/output hashes, record counts, and shape, not model quality.

In [8]:
_,_,RUN2=run_pipeline_timeseries()
for key in ['source_path','source_sha256','output_path','output_sha256','input_rows','output_rows','invalid_timestamps','missing_readings','processing']:
    assert RUN1[key]==RUN2[key],f'Non-repeatable: {key}'
assert source_hash(PIPELINE_OUT)==RUN1['output_sha256']
assert json.loads(MANIFEST_PATH.read_text())['records'][0]['source_sha256']==RUN1['source_sha256']
print('REPEATABILITY: PASS')
print('PROVENANCE: PASS')
print('Prepared output:',PIPELINE_OUT)
print('Part B manifest:',MANIFEST_PATH)
print('Part C manifest:',PIPELINE_MANIFEST)

REPEATABILITY: PASS
PROVENANCE: PASS
Prepared output: /content/week3_project/data/pipeline_v1/timeseries/prepared_timeseries.csv
Part B manifest: /content/week3_project/data/week3_manifest.json
Part C manifest: /content/week3_project/data/pipeline_v1/timeseries/pipeline_manifest.json


## Export / submission evidence
Download the completed notebook (File → Download → .ipynb), the prepared CSV and both manifests. Do not upload the entire raw dummy pack into a public repository unless redistribution is permitted. The existing GitHub repository must include `docs/week3_data_inventory.md`, `docs/week3_pipeline_diagram.png`, `docs/week3_pipeline_decisions.md`, `notebooks/week3_pipeline.ipynb`, `data/week3_manifest.json`, `data/processed/prepared_timeseries.csv`, and `data/pipeline_v1/timeseries/` run outputs.